# Italy Blocks Reproductive Rights Websites

This notebook produces the measurement findings behind the report "Italy Blocks Reproductive Rights Websites Women on Web (WoW) and Women Help Women (WHW)".

It analyses OONI Web Connectivity data collected in Italy (probe_cc = 'IT') between 2025-11-01 and 2026-06-01 for two domains: www.womenonweb.org and womenhelp.org

In [ ]:
import numpy as np
import pandas as pd
import altair as alt
import seaborn as sns
from tqdm import tqdm
tqdm.pandas()

In [ ]:
pd.options.display.max_columns = 500
pd.options.display.max_rows = 500
alt.data_transformers.disable_max_rows()

In [ ]:
from clickhouse_driver import Client as Clickhouse
from uuid import uuid4
from pathlib import Path

def click_query(q, params=None):
    click = Clickhouse("localhost")
    query_id = f"oonidata-{uuid4()}"
    print(f"Starting query_id: {query_id}")
    return click.query_dataframe(q, params=params, query_id=query_id)

In [ ]:
# see: https://learn.microsoft.com/en-us/windows/win32/winsock/windows-sockets-error-codes-2
unknown_failure_map = {
    ': server misbehaving': 'dns_server_misbehaving',
    ': read: connection refused': 'connection_refused',
    ': connect: network is unreachable': 'network_unreachable',
    'tls: first record does not look like a TLS handshake': 'tls_bad_first_record',
    'remote error: tls: handshake failure': 'tls_handshake_failure',
    'remote error: tls: illegal parameter': 'tls_illegal_parameter',
    'connectex: No connection could be made because the target machine actively refused it': 'connection_refused',
    'read: connection refused': 'connection_refused',
    'remote error: tls: access denied': 'tls_access_denied',
    'remote error: tls: internal error': 'tls_internal_error',
    'HTTP/1.x transport connection broken: malformed HTTP version': 'http_malformed_response',
    'net/http: timeout awaiting response headers': 'http_timeout',
    'read: operation timed out': 'timed_out',
    'connect: operation timed out': 'timed_out',
    ': No address associated with hostname': 'dns_nxdomain_error',
    ': connect: bad file descriptor': 'bad_file_descriptor',
    'stream error: stream ID': 'http_stream_error',
    
    # This looks more like a golang-bug: https://github.com/golang/go/issues/31259
    'readLoopPeekFailLocked: <nil>': 'http_golang_bug',

    ': connect: no route to host': 'host_unreachable',
    ': connect: cannot assign requested address': 'address_not_available',
    'getaddrinfow: The requested name is valid, but no data of the requested type was found.': 'dns_no_answer',
    'wsarecv: Se ha forzado la interrupción de una conexión existente por el host remoto.': 'connection_reset',
    'wsarecv: An existing connection was forcibly closed by the remote host.': 'connection_reset',
    'wsarecv: Connessione in corso interrotta forzatamente dall\'host remoto.': 'connection_reset',
    'wsarecv: Uma ligação existente foi forçada a fechar pelo anfitrião remoto': 'connection_reset',
    
    'getaddrinfow: Ceci est habituellement une erreur temporaire qui se produit durant la résolution du nom d’hôte et qui signifie que le serveur local n’a pas reçu de réponse d’un serveur faisant autorité': 'dns_temporary_failure',
    'getaddrinfow: Dies ist normalerweise ein zeitweiliger Fehler bei der Auflösung von Hostnamen. Grund ist, dass der lokale Server keine Rückmeldung vom autorisierenden Server erhalten hat.': 'dns_temporary_failure',
    'getaddrinfow: Este é geralmente um erro temporário durante a resolução de nomes de anfitrião e significa que o servidor local não recebeu uma resposta de um servidor autoritário': 'dns_temporary_failure',
    'getaddrinfow: Éste es normalmente un error temporal durante la resolución de nombres de host y significa que el servidor local no recibió una respuesta de un servidor autoritativo': 'dns_temporary_failure',
}
def map_unknown_failure(failure_str):
    if not failure_str.startswith("unknown_failure"):
        return failure_str
    for substring, clean_failure in unknown_failure_map.items():
        if substring in failure_str:
            return clean_failure
    return "unknown_failure"

def simplify_failure(failure_str):
    if failure_str in ['timed_out', 'generic_timeout_error', 'deferred_timeout_error']:
        return 'timeout'
    
    if failure_str in ['android_dns_cache_no_data', 'dns_nxdomain_error']:
        return 'nxdomain'
    
    if failure_str in ['connection_refused', 'connection_refused_error']:
        return 'connection_refused'
        
    return failure_str

ipv6_failures = ['address_not_available', 'address_family_not_supported', 'network_unreachable', 'host_unreachable']
def compute_analysis(row):
    failure_str = map_unknown_failure(row['failure_str_raw'])

    #if not pd.isnull(row['dns_answer']) and row['dns_answer'] in known_block_ips:
    #    return 'dns.confirmed'
        
    if row['tls_is_certificate_valid'] == True:
        return 'ok'

    if row['failure_class'] == 'ok':
        return 'ok'

    #if row['dns_consistency'] == 'inconsistent':
    #    return 'dns.inconsistent'
    
    if row['ip_as_org_name'] == 'Bogon':
        return 'dns.bogon'
    
    #if row['dns_blocking_scope'] not in ('u', 'n'):
    #    return f"dns.{row['dns_blocking_scope']}"
    
    if row['tls_is_certificate_valid'] == False:
        return 'tls.bad_cert'
    
    simple_failure = simplify_failure(failure_str)
    if simple_failure in ipv6_failures and pd.notna(row['ip']) and ':' in row['ip']:
        return 'ipv6_error'

    if simple_failure.startswith("ssl_"):
        simple_failure = 'bad_cert'
    
    prefix = row['failure_class']
    if prefix == 'https' and simple_failure.startswith('dns_') or simple_failure == 'nxdomain':
        prefix = 'dns'

    return f'{prefix}.{simple_failure}'

In [ ]:
def add_ooni_logo(chart, left_offset, top_offset):
    ooni_logo = alt.Chart(
        {"values": [{"url": "https://raw.githubusercontent.com/ooni/design-system/refs/heads/master/svgs/logos/OONI-HorizontalMonochrome.svg"}]}
    ).mark_image(opacity=0.5).encode(
        x=alt.value(left_offset), x2=alt.value(left_offset+80),  # pixels from left
        y=alt.value(top_offset), y2=alt.value(top_offset+40),    # pixels from top
        url="url:N"
    )

    return alt.vconcat(chart, ooni_logo).configure_concat(
            spacing=-30
        ).configure_view(
            strokeOpacity=0
        )

In [ ]:
MEASUREMENT_START_DAY = '2025-11-01'
MEASUREMENT_END_DAY = '2026-06-01'
ANALYSIS_COUNTRY_CODES = [
    "IT"
]
COUNTRY_NAME = "Italy"

In [ ]:
df_fp_dns = pd.read_csv('https://raw.githubusercontent.com/ooni/blocking-fingerprints/main/fingerprints_dns.csv')
known_block_ips = list(df_fp_dns['pattern'])

### www.womenonweb.org

In [ ]:
ANALYSIS_DOMAINS = """www.womenonweb.org""".split("\n")

In [ ]:
%%time
df = click_query("""
WITH multiIf(
    dns_failure IS NOT NULL, tuple('dns', dns_failure),
    tcp_failure IS NOT NULL, tuple('tcp', tcp_failure),
    tls_failure IS NOT NULL, tuple('tls', tls_failure),
    http_failure IS NOT NULL, tuple('https', http_failure),
    tuple('ok', '')
) as failure
SELECT 
report_id,
input,
software_name,
measurement_uid,
probe_cc,
probe_asn,
probe_as_org_name,
probe_as_cc,
network_type,
measurement_start_time,
hostname,
ip,
port,
ip_asn,
ip_as_org_name,
resolver_ip,
resolver_cc,
resolver_asn,
resolver_as_org_name,
resolver_as_cc,
dns_engine,
dns_failure,
dns_answer,
tcp_success,
tcp_failure,
tls_handshake_time,
tls_handshake_read_count,
tls_handshake_write_count,
tls_handshake_read_bytes,
tls_handshake_write_bytes,
tls_handshake_last_operation,
tls_cipher_suite IS NOT NULL as tls_success,
tls_is_certificate_valid,
tls_end_entity_certificate_subject_common_name,
tls_end_entity_certificate_issuer,
tls_end_entity_certificate_issuer_common_name,
tls_end_entity_certificate_san_list,
tls_end_entity_certificate_not_valid_after,
tls_end_entity_certificate_not_valid_before,
tls_certificate_chain_length,
tls_failure,
http_request_url,
http_failure,
http_runtime,
failure.1 as failure_class,
IF(failure_class = 'ok', 'ok', concat(failure_class, '.', failure_str)) as failure_str_full,
IF(startsWith(failure.2, 'unknown_failure'), 'unknown_failure', failure.2) as failure_str,
failure.2 as failure_str_raw
FROM obs_web
WHERE measurement_start_time > %(measurement_start_day)s
AND measurement_start_time < %(measurement_end_day)s
AND hostname IN %(domain_list)s
AND probe_cc IN %(cc_list)s
""", params={
    "measurement_start_day": MEASUREMENT_START_DAY,
    "measurement_end_day": MEASUREMENT_END_DAY,
    "domain_list": ANALYSIS_DOMAINS,
    "cc_list": ANALYSIS_COUNTRY_CODES,
})

In [ ]:
df = df[~df['ip'].str.contains('::', na=False)]
df = df[~df['ip'].str.contains(':', na=False)]

In [ ]:
df['analysis'] = df.progress_apply(compute_analysis, axis=1)

In [ ]:
has_tls = df["failure_class"].eq("tls").groupby(df["measurement_uid"]).transform("any")
mask = (df["failure_class"] == "https") & has_tls

df.loc[mask, "failure_class"] = "tls"
df.loc[mask, "analysis"] = df.loc[mask, "analysis"].str.replace("^https\\.", "tls.", regex=True)

# remove https observations where we receive a dns bogon to prevent double counting
has_dns_bogon = (
    df["analysis"]
    .eq("dns.bogon")
    .groupby(df["measurement_uid"])
    .transform("any")
)

drop_mask = has_dns_bogon & df["analysis"].eq("https.connection_refused")

df = df.loc[~drop_mask].copy()

In [ ]:
df_agg = df[[
    'measurement_start_time',
    'probe_as_org_name',
    'analysis',
    'hostname',
    'probe_asn',
    'ip',
    'resolver_asn',
    'resolver_as_org_name',
    'resolver_ip',
    'network_type',
    'measurement_uid'
]].groupby([
    pd.Grouper(freq='w', key='measurement_start_time'),
    'probe_asn',
    'hostname',
    'analysis',
    'probe_as_org_name',
    'resolver_asn',
    'resolver_as_org_name',
    'ip'
]).count().reset_index().rename(columns={'measurement_uid': 'obs_count'})

In [ ]:
top_analysis = (
    df_agg
    .groupby("analysis")["obs_count"]
    .sum()
    .sort_values(ascending=False)
)
print(top_analysis)
top_analysis = top_analysis.head(6).index.tolist()
top_analysis

In [ ]:
palette = [
    "#fcc419", "#ff8300", "#e03131", "#be0aff", "#1c7ed6", "#f06595"
]
colors = {}
palette_idx = 0
for a in top_analysis:
    if a == "ok":
        colors[a] = "#37b24d"  # green
    else:
        colors[a] = palette[palette_idx % len(palette)]
        palette_idx += 1

In [ ]:
df_top = df_agg[df_agg['analysis'].isin(top_analysis)]
main_chart = alt.Chart(df_top).mark_bar(point=True, size=10).encode(
    x="measurement_start_time:T",
    y=alt.Y("obs_count"),
    color=alt.Color(
        "analysis:N",
        scale=alt.Scale(
            domain=top_analysis,
            range=[colors[k] for k in top_analysis]
        ),
        legend=alt.Legend(title="Analysis")
    ),
    tooltip=["measurement_start_time", "obs_count", "analysis", "resolver_asn", "hostname", "probe_asn"]
).properties(
    width=600, 
    height=300,
    title=alt.TitleParams(
        text=f"Measurements for www.womenonweb.org in {COUNTRY_NAME}, Nov'25 - Jun'26",
        anchor="middle",
        orient="bottom",
        offset=10 
    )
).resolve_scale(
    y='independent'
)

add_ooni_logo(main_chart, -20, 0)

In [ ]:
ASN = [
    12874, # Fastweb 
    35612, # EOLO S.p.a
    30722, # Vodafone Italia
    29447, # Scaleway
    41497, # INTRED
    210278 # Sky Italia S.r.l
]

In [ ]:
df_agg['is_blocked_asn'] = df_agg['probe_asn'].isin(ASN) 

In [ ]:
df_top = df_agg[df_agg['analysis'].isin(top_analysis)]
main_chart = alt.Chart(df_top).mark_bar(point=True, size=10).encode(
    x="measurement_start_time:T",
    y=alt.Y("obs_count"),
    row=alt.Row("is_blocked_asn", title="ASN group"),
    color=alt.Color(
        "analysis:N",
        scale=alt.Scale(
            domain=top_analysis,
            range=[colors[k] for k in top_analysis]
        ),
        legend=alt.Legend(title="Analysis")
    ),
    tooltip=["measurement_start_time", "obs_count", "analysis", "resolver_asn", "hostname", "probe_asn"]
).properties(
    width=600, 
    height=300,
    title=alt.TitleParams(
        text=f"Blocked vs Accessible ASNs (www.womenonweb.org) in {COUNTRY_NAME}, Nov'25 - Jun'26",
        anchor="middle",
        orient="bottom",
        offset=10 
    )
).resolve_scale(
    y='independent'
)

add_ooni_logo(main_chart, -100, 10)

In [ ]:
DNS_ASN = {
    12874: "Fastweb SpA",
    35612: "EOLO S.p.A.",
    30722: "Vodafone Italia S.p.A.",
    210278: "Sky Italia srl"
}

In [ ]:
df_top = df_agg[df_agg['analysis'].isin(top_analysis)]
df_top = df_top[df_top['probe_asn'].isin(DNS_ASN)]

for asn, isp_name in DNS_ASN.items():
    df_chart = df_top[(df_top['probe_asn'] == df_top['resolver_asn']) & (df_top['probe_asn'] == asn)]
    main_chart = alt.Chart(df_chart).mark_bar(point=True, size=10).encode(
        x="measurement_start_time:T",
        y=alt.Y("obs_count"),
        row=alt.Row("probe_asn"),
        column=alt.Column("resolver_asn"),
        color=alt.Color(
            "analysis:N",
            scale=alt.Scale(
                domain=top_analysis,
                range=[colors[k] for k in top_analysis]
            ),
            legend=alt.Legend(title="Analysis")
        ),
        tooltip=["measurement_start_time", "obs_count", "analysis", "resolver_asn", "hostname", "probe_asn"]
    ).properties(
        width=600, 
        height=300,
        title=alt.TitleParams(
            text=f"Measurements for www.womenonweb.org on {isp_name} (AS{asn}) when using ISP resolver, Nov'25 - June'26",
            anchor="middle",
            orient="bottom",
            offset=10 
        )
    ).resolve_scale(
        y='independent'
    )
    
    main_chart = add_ooni_logo(main_chart, -100, 10)
    main_chart.save(f"2026 Italy Reprodcutive Rights/charts/womenonweb_as_resolver_{asn}.png")

In [ ]:
BAD_CERT_ASN = [
    210278
]

In [ ]:
df_top = df_agg[df_agg['analysis'].isin(top_analysis)]
df_top = df_top[df_top['probe_asn'].isin(BAD_CERT_ASN)]

main_chart = alt.Chart(df_top).mark_bar(point=True, size=10).encode(
    x="measurement_start_time:T",
    y=alt.Y("obs_count"),
    row=alt.Row("ip"),
    color=alt.Color(
        "analysis:N",
        scale=alt.Scale(
            domain=top_analysis,
            range=[colors[k] for k in top_analysis]
        ),
        legend=alt.Legend(title="Analysis")
    ),
    tooltip=["measurement_start_time", "obs_count", "analysis", "resolver_asn", "hostname", "probe_asn"]
).properties(
    width=600, 
    height=300,
    title=alt.TitleParams(
        text=f"Measurement for www.womenonweb.org on Sky Italia srl (AS210278), Nov'25 - Jun'26",
        anchor="middle",
        orient="bottom",
        offset=10 
    )
).resolve_scale(
    y='independent'
)

add_ooni_logo(main_chart, -100, 10)

In [ ]:
EXTRA_ASN = {
    29447: 'Scaleway SAS',
    41497: 'INTRED S.p.A.'
}

In [ ]:
df_top = df_agg[df_agg['analysis'].isin(top_analysis)]
df_top = df_top[df_top['probe_asn'].isin(EXTRA_ASN)]

main_chart = alt.Chart(df_top).mark_bar(point=True, size=10).encode(
    x="measurement_start_time:T",
    y=alt.Y("obs_count"),
    row=alt.Row("probe_asn"),
    color=alt.Color(
        "analysis:N",
        scale=alt.Scale(
            domain=top_analysis,
            range=[colors[k] for k in top_analysis]
        ),
        legend=alt.Legend(title="Analysis")
    ),
    tooltip=["measurement_start_time", "obs_count", "analysis", "resolver_asn", "hostname", "probe_asn"]
).properties(
    width=600, 
    height=300,
    title=alt.TitleParams(
        text=f"Measurements for www.womenonweb.org on AS29447 ({DNS_ASN[29447]}), AS41497 ({DNS_ASN[41497]}), Nov'25 - June'26",
        anchor="middle",
        orient="bottom",
        offset=10 
    )
).resolve_scale(
    y='independent'
)
    
main_chart = add_ooni_logo(main_chart, -100, 10)
main_chart

### womenhelp.org

In [ ]:
ANALYSIS_DOMAINS = """womenhelp.org""".split("\n")

In [ ]:
%%time
df = click_query("""
WITH multiIf(
    dns_failure IS NOT NULL, tuple('dns', dns_failure),
    tcp_failure IS NOT NULL, tuple('tcp', tcp_failure),
    tls_failure IS NOT NULL, tuple('tls', tls_failure),
    http_failure IS NOT NULL, tuple('https', http_failure),
    tuple('ok', '')
) as failure
SELECT 
report_id,
input,
software_name,
measurement_uid,
probe_cc,
probe_asn,
probe_as_org_name,
probe_as_cc,
network_type,
measurement_start_time,
hostname,
ip,
port,
ip_asn,
ip_as_org_name,
resolver_ip,
resolver_cc,
resolver_asn,
resolver_as_org_name,
resolver_as_cc,
dns_engine,
dns_failure,
dns_answer,
tcp_success,
tcp_failure,
tls_handshake_time,
tls_handshake_read_count,
tls_handshake_write_count,
tls_handshake_read_bytes,
tls_handshake_write_bytes,
tls_handshake_last_operation,
tls_cipher_suite IS NOT NULL as tls_success,
tls_is_certificate_valid,
tls_end_entity_certificate_subject_common_name,
tls_end_entity_certificate_issuer,
tls_end_entity_certificate_issuer_common_name,
tls_end_entity_certificate_san_list,
tls_end_entity_certificate_not_valid_after,
tls_end_entity_certificate_not_valid_before,
tls_certificate_chain_length,
tls_failure,
http_request_url,
http_failure,
http_runtime,
failure.1 as failure_class,
IF(failure_class = 'ok', 'ok', concat(failure_class, '.', failure_str)) as failure_str_full,
IF(startsWith(failure.2, 'unknown_failure'), 'unknown_failure', failure.2) as failure_str,
failure.2 as failure_str_raw
FROM obs_web
WHERE measurement_start_time > %(measurement_start_day)s
AND measurement_start_time < %(measurement_end_day)s
AND hostname IN %(domain_list)s
AND probe_cc IN %(cc_list)s
""", params={
    "measurement_start_day": MEASUREMENT_START_DAY,
    "measurement_end_day": MEASUREMENT_END_DAY,
    "domain_list": ANALYSIS_DOMAINS,
    "cc_list": ANALYSIS_COUNTRY_CODES,
})

In [ ]:
df = df[~df['ip'].str.contains('::', na=False)]
df = df[~df['ip'].str.contains(':', na=False)]

In [ ]:
df['analysis'] = df.progress_apply(compute_analysis, axis=1)

In [ ]:
has_tls = df["failure_class"].eq("tls").groupby(df["measurement_uid"]).transform("any")
mask = (df["failure_class"] == "https") & has_tls

df.loc[mask, "failure_class"] = "tls"
df.loc[mask, "analysis"] = df.loc[mask, "analysis"].str.replace("^https\\.", "tls.", regex=True)

has_dns_bogon = (
    df["analysis"]
    .eq("dns.bogon")
    .groupby(df["measurement_uid"])
    .transform("any")
)

drop_mask = has_dns_bogon & df["analysis"].eq("https.connection_refused")

df = df.loc[~drop_mask].copy()

In [ ]:
df_agg = df[[
    'measurement_start_time',
    'probe_as_org_name',
    'analysis',
    'hostname',
    'probe_asn',
    'resolver_asn',
    'resolver_as_org_name',
    'resolver_ip',
    'network_type',
    'measurement_uid'
]].groupby([
    pd.Grouper(freq='w', key='measurement_start_time'),
    'probe_asn',
    'hostname',
    'analysis',
    'probe_as_org_name',
    'resolver_asn',
    'resolver_as_org_name',
]).count().reset_index().rename(columns={'measurement_uid': 'obs_count'})

In [ ]:
top_analysis = (
    df_agg
    .groupby("analysis")["obs_count"]
    .sum()
    .sort_values(ascending=False)
)
print(top_analysis)
top_analysis = top_analysis.head(6).index.tolist()
top_analysis

In [ ]:
palette = [
    "#fcc419", "#ff8300", "#e03131", "#be0aff", "#1c7ed6", "#f06595"
]
colors = {}
palette_idx = 0
for a in top_analysis:
    if a == "ok":
        colors[a] = "#37b24d"  # green
    else:
        colors[a] = palette[palette_idx % len(palette)]
        palette_idx += 1

In [ ]:
df_top = df_agg[df_agg['analysis'].isin(top_analysis)]

main_chart = alt.Chart(df_top).mark_bar(point=True, size=10).encode(
    x="measurement_start_time:T",
    y=alt.Y("obs_count"),
    row=alt.Column("probe_asn"),
    color=alt.Color(
        "analysis:N",
        scale=alt.Scale(
            domain=top_analysis,
            range=[colors[k] for k in top_analysis]
        ),
        legend=alt.Legend(title="Analysis")
    ),
    tooltip=["measurement_start_time", "obs_count", "analysis", "resolver_asn", "hostname", "probe_asn"]
).properties(
    width=600, 
    height=300,
    title=alt.TitleParams(
        text=f"Measurements for womenhelp.org in {COUNTRY_NAME}, Nov'25 - Jun'26",
        anchor="middle",
        orient="bottom",
        offset=10 
    )
).resolve_scale(
    y='independent'
)

add_ooni_logo(main_chart, -20, 0)

In [ ]:
DNS_ASN = {
    12874: "Fastweb SpA",
    29447: "Scaleway SAS",
    1267: "WIND TRE S.P.A.",
    35612: "EOLO S.p.A.",
    30722: "Vodafone Italia S.p.A.",
    210278: "Sky Italia srl"
}

In [ ]:
for asn, isp_name in DNS_ASN.items():
    df_chart = df_top[(df_top['probe_asn'] == df_top['resolver_asn']) & (df_top['probe_asn'] == asn)]
    
    main_chart = alt.Chart(df_chart).mark_bar(point=True, size=10).encode(
        x="measurement_start_time:T",
        y=alt.Y("obs_count"),
        row=alt.Row("probe_asn"),
        column=alt.Column("resolver_asn"),
        color=alt.Color(
            "analysis:N",
            scale=alt.Scale(
                domain=top_analysis,
                range=[colors[k] for k in top_analysis]
            ),
            legend=alt.Legend(title="Analysis")
        ),
        tooltip=["measurement_start_time", "obs_count", "analysis", "resolver_asn", "hostname", "probe_asn"]
    ).properties(
        width=600, 
        height=300,
        title=alt.TitleParams(
            text=f"Measurements for womenhelp.org on {isp_name} ({asn}) when using ISP resolver, Nov'25 - June'26",
            anchor="middle",
            orient="bottom",
            offset=10 
        )
    ).resolve_scale(
        y='independent'
    )
    
    main_chart = add_ooni_logo(main_chart, -100, 10)
    main_chart.save(f"charts/womenhelp_as_isp_{asn}.png")

In [ ]:
BAD_CERT_ASN = [
    210278
]

In [ ]:
df_top = df_agg[df_agg['analysis'].isin(top_analysis)]
df_top = df_top[df_top['probe_asn'].isin(BAD_CERT_ASN)]

main_chart = alt.Chart(df_top).mark_bar(point=True, size=10).encode(
    x="measurement_start_time:T",
    y=alt.Y("obs_count"),
    row=alt.Row("ip"),
    color=alt.Color(
        "analysis:N",
        scale=alt.Scale(
            domain=top_analysis,
            range=[colors[k] for k in top_analysis]
        ),
        legend=alt.Legend(title="Analysis")
    ),
    tooltip=["measurement_start_time", "obs_count", "analysis", "resolver_asn", "hostname", "probe_asn"]
).properties(`
    width=600, 
    height=300,
    title=alt.TitleParams(
        text=f"Measurement for www.womenonweb.org on Sky Italia srl (AS210278), Nov'25 - Jun'26",
        anchor="middle",
        orient="bottom",
        offset=10 
    )
).resolve_scale(
    y='independent'
)

add_ooni_logo(main_chart, -100, 10)